# Notebook 5: Phyloseq Object Construction
In this notebook, we will integrate our core microbiome data components into a single **phyloseq** object:
1. **ASV Table**: `seqtab_final.rds` (Abundance of 6,013 ASVs across 558 samples).
2. **Taxonomy Table**: `taxa_final.rds` (Taxonomic classifications from Kingdom to Genus).
3. **Sample Metadata**: `master_metadata_final.tsv` (Experimental design and clinical variables).

In [3]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
# 1. Create a local library directory inside the project folder
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
if (!dir.exists(local_lib)) {
    dir.create(local_lib, recursive = TRUE)
}

# 2. Tell R to use this local library path
.libPaths(c(local_lib, .libPaths()))
cat("Current library paths:\n")
print(.libPaths())

# 3. Install BiocManager and packages into the local library
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos = "https://cloud.r-project.org", lib = local_lib)

cat("Installing phyloseq and tidyverse locally...\n")
BiocManager::install(c("phyloseq", "tidyverse"), lib = local_lib, ask = FALSE, update = FALSE)

cat("Local installation completed successfully!\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> # 1. Create a local library directory inside the project folder
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> if (!dir.exists(local_lib)) {
+     dir.create(local_lib, recursive = TRUE)
+ }
> 
> # 2. Tell R to use this local library path
> .libPaths(c(local_lib, .libPaths()))
> cat("Current library

In [4]:
import os
import subprocess

project_dir = "/home/azureuser/Microbiome_project"
container_path = os.path.join(project_dir, "dada2.sif")

r_code = """
# 1. Point to the local library directory where packages were installed
local_lib <- "/home/azureuser/Microbiome_project/R_libs"
.libPaths(c(local_lib, .libPaths()))

# 2. Load required libraries
library(phyloseq)
library(tidyverse)

# 3. Load components
cat("Loading ASV table, taxonomy, and metadata...\n")
seqtab.nochim <- readRDS("seqtab_final.rds")
taxa <- readRDS("taxa_final.rds")
metadata <- read.delim("master_metadata_final.tsv", header = TRUE, sep = "\\t")

# 4. Match sample names between ASV table and metadata
rownames(metadata) <- metadata$Run
common.samples <- intersect(rownames(seqtab.nochim), rownames(metadata))
seqtab.nochim <- seqtab.nochim[common.samples, ]
metadata <- metadata[common.samples, ]

# 5. Create Phyloseq components
ps_otu <- otu_table(seqtab.nochim, taxa_are_rows = FALSE)
ps_tax <- tax_table(taxa)
ps_meta <- sample_data(metadata)

# 6. Merge into a single phyloseq object
ps <- phyloseq(ps_otu, ps_tax, ps_meta)

cat("\\n--- Phyloseq Object Summary ---\\n")
print(ps)

# 7. Save the final phyloseq object for downstream analysis
saveRDS(ps, "phyloseq_final.rds")
cat("\\nPhyloseq object successfully created and saved to phyloseq_final.rds!\\n")
"""

cmd = ["apptainer", "exec", container_path, "R", "-e", r_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
  print("Warnings/Errors:", result.stderr)


R version 4.4.2 (2024-10-31) -- "Pile of Leaves"
Copyright (C) 2024 The R Foundation for Statistical Computing
Platform: x86_64-pc-linux-gnu

R is free software and comes with ABSOLUTELY NO WARRANTY.
You are welcome to redistribute it under certain conditions.
Type 'license()' or 'licence()' for distribution details.

  Natural language support but running in an English locale

R is a collaborative project with many contributors.
Type 'contributors()' for more information and
'citation()' on how to cite R or R packages in publications.

Type 'demo()' for some demos, 'help()' for on-line help, or
'help.start()' for an HTML browser interface to help.
Type 'q()' to quit R.

> 
> # 1. Point to the local library directory where packages were installed
> local_lib <- "/home/azureuser/Microbiome_project/R_libs"
> .libPaths(c(local_lib, .libPaths()))
> 
> # 2. Load required libraries
> library(phyloseq)
> library(tidyverse)
> 
> # 3. Load components
> cat("Loading ASV table, taxonomy, and met